***Sound Event Detection***  
This notebook uses the panns_inference and webrtcvad libraries to explore the various sonic elements of a peice of audio. It particularly focuses on the detection and location of human speech within audio. This notebook was designed to assist exploration of audio recordings from the PARADISEC collection.

This code loads in the .wav file which is to be analyzed, then does the majority of the data processing. It ultimately creates the audio_tags and speech_tags lists, which contain audio labels for each second of the audio.

In [ ]:
import librosa
import panns_inference
import numpy as np
import webrtcvad
from pydub import AudioSegment
audio_file_path="/content/AC1-018-A (1).wav" #Include the file path to the .wav to be analyzed
(audio,sr)=librosa.core.load(audio_file_path,sr=48000,mono=True)
chunk_duration=1
chunk_samples=int(sr*chunk_duration)
chunks = [audio[i:i + chunk_samples] for i in range(0, len(audio)-len(audio)%chunk_samples, chunk_samples)]
audio_tags=[]
speech_tags=[]
at=AudioTagging(checkpoint_path=None, device='cuda')
vad=webrtcvad.Vad()
for i in chunks:
  inp16 = (i * 32767).astype(np.int16)
  inp = i[None, :]
  num_splits=100
  rt=0
  for j in range(num_splits):
    temp=inp16[j*sr*chunk_duration//num_splits:(j+1)*sr*chunk_duration//num_splits]
    if vad.is_speech(temp.tobytes(),sr):
      rt+=1
  if rt>=50:
    speech_tags.append(True)
  else:
    speech_tags.append(False)
  chunkwise_output,embedding=at.inference(inp)
  scores = chunkwise_output[0]
  top_indices = np.argsort(scores)[::-1][:6]
  audio_tags.append([])
  for j in top_indices:
    audio_tags[-1].append((labels[j],scores[j]))

This code returns the audio labels and speech detection for each requested second of the audio. Enter a range of seconds (ie. 0-5, 52-60) to see the information for each.

In [ ]:
high=0
low=0
while not (low<high):
  indices=input("Which range of seconds would you like information for? (ie. 0-10, 20-22): ")
  low,high=indices.split("-")
  low=int(low)
  high=int(high)
  if not (low<high):
    print("Not a valid range")
for i in range(low,high+1):
  print("Second "+str(i)+": ")
  for j in audio_tags[i]:
    print(j[0]+": "+str(j[1]))
  print("VAD speech detected: "+str(speech_tags[i]))
  print()


This code uses the previously created tags to create a "smart detection" of speech in the audio file. By combining the speech data recieved from the VAD and SED, it predicts whether each second of audio contains speech. By combining the data from two sources, this method provides a more accurate prediction then either source individually.

In [ ]:
speech=[]
for i in range(len(audio_tags)):
  speech.append(0)
  rs=0
  for j in range(len(audio_tags[i])):
    if audio_tags[i][j][0] in ["Speech","Narration, monologue","Male speech, man speaking","Female speech, woman speaking","Mantra"]:
      if audio_tags[i][j][1]>0.6 or j==0:
        speech[-1]=2
        break
      if j==1 and audio_tags[i][0][1]<0.4 and audio_tags[i][j][1]>.1:
        speech[-1]=1
      rs+=audio_tags[i][j][1]
  if rs>=1.2:
    speech[-1]=2
  elif speech_tags[i]:
    speech[-1]=2
  elif rs>=.8 and speech[-1]==0:
    speech[-1]=1
final=[]
if (speech[0]==1 and speech[1]==2) or speech[0]==2:
  final.append(True)
else:
  final.append(False)
for i in range(1,len(speech)-1):
  if speech[i]==2:
    final.append(True)
  elif speech[i]==1 and (speech[i-1]+speech[i+1]>=3):
    final.append(True)
  else:
    final.append(False)
if speech[-1]==2 or (speech[-1]==1 and speech[-2]==2):
  final.append(True)
else:
  final.append(False)
for i in range(len(final)):
  print(str(i)+" "+str(final[i]))




This code does further label prediction on the audio, structuring the resulting tags in a way that allows the tags to be graphed over time.

In [ ]:
import matplotlib.pyplot as plt
sed = SoundEventDetection(checkpoint_path=None, device='cuda')
audioin=audio[None, :]
framewise_output = sed.inference(audioin)

Graphs the confidence of the five most prevalent audio tags over the course of the audio. Saves the graph as "output_graph.png"

In [ ]:
output = framewise_output[0]
top_n = 5
time = np.arange(output.shape[0])/48000*320
plt.figure(figsize=(max(round(max(time)/10),7), 5))
top_classes = np.argsort(output.max(axis=0))[::-1][:top_n]
for i in top_classes:
    label = sed.labels[i]
    plt.plot(time, output[:, i], label=label)
plt.legend()
plt.title("Audio Tag Confidence Over Time")
plt.xlabel("Time (s)")
plt.ylabel("Label Confidence")
plt.savefig("output_graph.png")